In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, kendalltau, rankdata, entropy
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
import joblib
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/metadata.json
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/study_final.pkl
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_oof_preds.npy
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_models_info.csv
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_test_preds.npy
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/trial_oof_preds.pkl
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/trial_predictions.pkl
/kaggle/input/pss6e2-artifacts-oofs2/optuna_artifacts/metadata.json
/kaggle/input/pss6e2-artifacts-oofs2/optuna_artifacts/study_final.pkl
/kaggle/input/pss6e2-artifacts-oofs2/optuna_artifacts/all_oof_preds.npy
/kaggle/input/pss6e2-artifacts-oofs2/optuna_artifacts/all_models_info.csv
/kaggle/input/pss6e2-artifacts-oofs2/optuna_artifacts/all_test_p

In [2]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

CONFIG = config()

In [3]:
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')
y = train['Heart Disease']
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')
sample_submission = pd.read_csv('/kaggle/input/playground-series-s6e2/sample_submission.csv')

In [4]:
# Load numpy arrays
oofs_gblinear = np.load('/kaggle/input/pss6e2-artifacts-oofs2/optuna_artifacts/all_oof_preds.npy')
test_gblinear = np.load('/kaggle/input/pss6e2-artifacts-oofs2/optuna_artifacts/all_test_preds.npy')

oofs_gbtree = np.load('/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_oof_preds.npy')
test_gbtree = np.load('/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_test_preds.npy')

print(f"gblinear shapes: OOF={oofs_gblinear.shape}, Test={test_gblinear.shape}")
print(f"gbtree shapes: OOF={oofs_gbtree.shape}, Test={test_gbtree.shape}")

oofs_gblinear_df = pd.DataFrame(oofs_gblinear.T)  
test_gblinear_df = pd.DataFrame(test_gblinear.T)

oofs_gbtree_df = pd.DataFrame(oofs_gbtree.T)
test_gbtree_df = pd.DataFrame(test_gbtree.T)

oofs_gblinear_df.columns = [f'gblinear_model_{i}' for i in range(oofs_gblinear_df.shape[1])]
test_gblinear_df.columns = [f'gblinear_model_{i}' for i in range(test_gblinear_df.shape[1])]

oofs_gbtree_df.columns = [f'gbtree_model_{i}' for i in range(oofs_gbtree_df.shape[1])]
test_gbtree_df.columns = [f'gbtree_model_{i}' for i in range(test_gbtree_df.shape[1])]

print(f"\nDataFrame shapes:")
print(f"OOF gblinear: {oofs_gblinear_df.shape}")
print(f"Test gblinear: {test_gblinear_df.shape}")
print(f"OOF gbtree: {oofs_gbtree_df.shape}")
print(f"Test gbtree: {test_gbtree_df.shape}")

print(f"\nFirst few columns:")
print(oofs_gblinear_df.iloc[:, :3].head())

gblinear shapes: OOF=(24, 630000), Test=(24, 270000)
gbtree shapes: OOF=(120, 630000), Test=(120, 270000)

DataFrame shapes:
OOF gblinear: (630000, 24)
Test gblinear: (270000, 24)
OOF gbtree: (630000, 120)
Test gbtree: (270000, 120)

First few columns:
   gblinear_model_0  gblinear_model_1  gblinear_model_2
0          0.591905          0.513477          0.511572
1          0.419465          0.454035          0.409088
2          0.424550          0.455011          0.418623
3          0.450497          0.466717          0.433526
4          0.591767          0.512654          0.490255


In [5]:
all_oofs_df = pd.concat([oofs_gblinear_df, oofs_gbtree_df], axis=1)
all_tests_df = pd.concat([test_gblinear_df, test_gbtree_df], axis=1)

print(f"Combined OOFs shape: {all_oofs_df.shape}")
print(f"Combined Tests shape: {all_tests_df.shape}")

print(f"\nModel types distribution:")
print(f"GBLinear models: {len([c for c in all_oofs_df.columns if 'gblinear' in c])}")
print(f"GBTree models: {len([c for c in all_oofs_df.columns if 'gbtree' in c])}")
print(f"Total models: {all_oofs_df.shape[1]}")

Combined OOFs shape: (630000, 144)
Combined Tests shape: (270000, 144)

Model types distribution:
GBLinear models: 24
GBTree models: 120
Total models: 144


In [6]:
test_preds = test_gbtree_df['gbtree_model_59'].values
sample_submission[CONFIG.TARGET] = test_preds
sample_submission.to_csv('submission_csv_peak_optuna.csv', index=False)

In [7]:
# from sklearn.model_selection import StratifiedKFold
# from sklearn.linear_model import Ridge, LogisticRegression
# from sklearn.preprocessing import StandardScaler, LabelEncoder
# from tqdm import tqdm
# import numpy as np

# # Your data
# X_meta_train = all_oofs_df.values
# X_meta_test = all_tests_df.values
# y = train[CONFIG.TARGET].map(class_mapping).values

# strat_cols = ['Thallium', 'Chest pain type', 'Heart Disease']
# le = LabelEncoder()
# stratify_feature = le.fit_transform(train[strat_cols].astype(str).agg('_'.join, axis=1))

# print(f"Meta-train shape: {X_meta_train.shape}")
# print(f"Meta-test shape: {X_meta_test.shape}")

# # Stratified K-Fold setup
# skf = StratifiedKFold(n_splits=CONFIG.N_FOLDS, shuffle=True, random_state=CONFIG.SEED)

# # Storage for predictions
# meta_oof_preds = np.zeros(len(X_meta_train))
# meta_test_preds = np.zeros(len(X_meta_test))
# fold_scores = []

# print(f"\nTraining meta-model with {CONFIG.N_FOLDS}-fold CV")
# print("="*60)

# # CV loop
# for fold, (train_idx, val_idx) in enumerate(skf.split(X_meta_train, stratify_feature), 1):
    
#     print(f"\nFold {fold}:")
#     print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}")
    
#     # Split meta-data
#     X_train_fold = X_meta_train[train_idx]
#     X_val_fold = X_meta_train[val_idx]
#     y_train_fold = y[train_idx]
#     y_val_fold = y[val_idx]
    
#     # ===== FIXED: USE RIDGE, NOT LOGISTIC (for now) =====
#     # Scale features
#     scaler = StandardScaler()
#     X_train_scaled = scaler.fit_transform(X_train_fold)
#     X_val_scaled = scaler.transform(X_val_fold)
#     X_test_scaled = scaler.transform(X_meta_test)
    
#     # ===== OPTION A: RIDGE REGRESSION (WORKS) =====
#     ridge = Ridge(alpha=0.01, random_state=CONFIG.SEED + fold)
#     ridge.fit(X_train_scaled, y_train_fold)
    
#     # Predict
#     val_preds = ridge.predict(X_val_scaled)
#     test_preds = ridge.predict(X_test_scaled)
    
#     # ===== OPTION B: FIXED LOGISTIC REGRESSION =====
#     # Uncomment this if you want to try logistic
#     # logreg = LogisticRegression(
#     #     C=0.01,  # STRONG REGULARIZATION (1/alpha)
#     #     penalty='l2',
#     #     solver='lbfgs',  # BETTER FOR LARGE DATASETS
#     #     max_iter=1000,
#     #     random_state=CONFIG.SEED + fold
#     # )
#     # logreg.fit(X_train_scaled, y_train_fold)
#     # val_preds = logreg.predict_proba(X_val_scaled)[:, 1]
#     # test_preds = logreg.predict_proba(X_test_scaled)[:, 1]
    
#     # Clip predictions to [0, 1]
#     val_preds = np.clip(val_preds, 0, 1)
#     test_preds = np.clip(test_preds, 0, 1)
    
#     # Check for weird predictions
#     print(f"  Val preds range: [{val_preds.min():.4f}, {val_preds.max():.4f}]")
#     print(f"  Mean val pred: {val_preds.mean():.4f}")
    
#     # Store predictions
#     meta_oof_preds[val_idx] = val_preds
#     meta_test_preds += test_preds / CONFIG.N_FOLDS
    
#     # Score
#     fold_score = roc_auc_score(y_val_fold, val_preds)
#     fold_scores.append(fold_score)
    
#     print(f"  Fold {fold} AUC: {fold_score:.6f}")
#     print(f"  Ridge coef stats: mean={ridge.coef_.mean():.6f}, std={ridge.coef_.std():.6f}")

# # ===== FINAL RESULTS =====
# print(f"\n{'='*60}")
# print("META-MODEL CV RESULTS")
# print(f"{'='*60}")

# print(f"Fold scores: {[f'{s:.6f}' for s in fold_scores]}")
# print(f"Mean fold score: {np.mean(fold_scores):.6f} (±{np.std(fold_scores):.6f})")

# # Check for weird predictions in final OOF
# print(f"\nFinal OOF predictions analysis:")
# print(f"  Range: [{meta_oof_preds.min():.6f}, {meta_oof_preds.max():.6f}]")
# print(f"  Mean: {meta_oof_preds.mean():.6f}")
# print(f"  Std: {meta_oof_preds.std():.6f}")

# # Final score
# final_score = roc_auc_score(y, meta_oof_preds)
# print(f"\nOOF AUC Score: {final_score:.6f}")

# # Compare with simple average
# simple_avg_preds = np.mean(X_meta_train, axis=1)
# simple_avg_score = roc_auc_score(y, simple_avg_preds)
# print(f"Simple average baseline: {simple_avg_score:.6f}")
# print(f"Meta-model improvement: +{final_score - simple_avg_score:.6f}")

# # Create submission
# sample_submission['Heart Disease'] = meta_test_preds
# sample_submission.to_csv(f'submission_meta_ridge_{final_score:.6f}.csv', index=False)
# print(f"\n✅ Saved: submission_meta_ridge_{final_score:.6f}.csv")